# Phi-2 FFN merge experiment (2 layers, low memory)

This notebook reproduces the **core idea** from `v5.py` on [microsoft/phi-2](https://huggingface.co/microsoft/phi-2):

1. Wrap each FFN (`fc1`, `fc2`) with **LoRA** so layers can diverge while sharing a base later.
2. Measure **pairwise base distance** between two FFN blocks on cached activations.
3. **Soft-align** the two representative bases (LM loss + align + anchor).
4. **Collapse** to one shared base by tying `fc1`/`fc2` weights and keeping per-layer LoRA.

To fit ~16 GB **CPU RAM**, we keep only **2 transformer layers** (default: original layers 0 and 1), use small batches/sequences, and reduced training steps.

**Environment:** use a kernel with `torch` + `transformers` (not `mxnet_env` unless you fix numpy). This notebook loads WikiText **without** the `datasets` package (avoids pandas/numpy conflicts).

**Model:** Phi-2 should already be in `~/.cache/huggingface/hub/` after download.

In [3]:
from pathlib import Path
cache = Path.home() / ".cache/huggingface/hub/models--microsoft--phi-2/snapshots"
snaps = sorted(cache.glob("*")) if cache.exists() else []
print("Phi-2:", snaps[-1] if snaps else "NOT FOUND")

/bin/bash: line 1: huggingface-cli: command not found


## Configuration

**Kernel:** use `base` or a fresh env — not `mxnet_env` (numpy 1.19 vs pandas 2.2). Then **Restart Kernel** and run all.

In [ ]:
_removed_pip_cell = True

In [ ]:
# Skip pip install — mxnet_env has numpy/pandas conflicts. This notebook needs no `datasets` package.

  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached pyarrow-21.0.0-cp39-cp39-manylinux_2_28_x86_64.whl.metadata (3.3 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.7.0-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py39-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
  Using cached aiohttp-3.13.5-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached typer-0.23.2-py3-none-any.whl.metadata (16 kB)
  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)


In [7]:
pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 2.9 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.3/791.3 kB 2.8 MB/s eta 0:00:003.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.6 MB/s eta 0:00:00 MB/s eta 0:00:01:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
import gc
import copy
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# -------------------------
# Low-memory smoke settings
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "microsoft/phi-2"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32  # fp32 on CPU is safer numerically

KEEP_LAYER_INDICES = [0, 1]   # which original Phi layers to keep
TARGET_CLUSTERS = 1           # merge 2 singleton clusters -> 1 shared FFN

BATCH_SIZE = 1
SEQ_LEN = 32
CALIB_BATCHES = 3

ALIGN_STEPS = 150
RECOVERY_STEPS = 40
LR_ALIGN = 3e-5
LR_RECOVERY = 1e-5
LAMBDA_MAX = 3.0
MU = 1.0
WARMUP_FRAC = 0.2
LORA_RANK = 4

THRESH_BAD = 0.8              # reject merge rollback threshold on delta L
EVAL_BATCHES = 5

print(f"Device: {DEVICE}, dtype: {DTYPE}")
print(f"Keeping layers: {KEEP_LAYER_INDICES}, target clusters: {TARGET_CLUSTERS}")

Device: cpu, dtype: torch.float32
Keeping layers: [0, 1], target clusters: 1


## Helpers (adapted from `v5.py` for Phi-2)

In [2]:
def get_layers(model: nn.Module) -> nn.ModuleList:
    return model.model.layers


def mlp_input(layer: nn.Module, h_raw: torch.Tensor) -> torch.Tensor:
    """Phi applies MLP on input_layernorm(hidden_states at block input)."""
    return layer.input_layernorm(h_raw)


def count_params_total(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def count_params_unique(model: nn.Module) -> int:
    seen, total = set(), 0
    for p in model.parameters():
        ptr = id(p)
        if ptr in seen:
            continue
        seen.add(ptr)
        total += p.numel()
    return total


def compression_ratio(before: int, after: int) -> float:
    return (1 - after / before) * 100 if before else 0.0


class LoRALinear(nn.Module):
    """output = base(x) + x @ A @ B"""

    def __init__(self, linear: nn.Linear, rank: int):
        super().__init__()
        self.linear = linear
        in_dim, out_dim = linear.in_features, linear.out_features
        dev, dt = linear.weight.device, linear.weight.dtype
        self.A = nn.Parameter(torch.randn(in_dim, rank, device=dev, dtype=dt) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, out_dim, device=dev, dtype=dt))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + (x @ self.A) @ self.B

    def base_forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


def wrap_mlp_with_lora(layer: nn.Module, rank: int) -> None:
    if not isinstance(layer.mlp.fc1, LoRALinear):
        layer.mlp.fc1 = LoRALinear(layer.mlp.fc1, rank)
    if not isinstance(layer.mlp.fc2, LoRALinear):
        layer.mlp.fc2 = LoRALinear(layer.mlp.fc2, rank)


def full_mlp_forward(layer: nn.Module, x: torch.Tensor) -> torch.Tensor:
    h = layer.mlp.fc1(x)
    h = layer.mlp.activation_fn(h)
    return layer.mlp.fc2(h)


def base_mlp_forward(layer: nn.Module, x: torch.Tensor) -> torch.Tensor:
    h = layer.mlp.fc1.base_forward(x)
    h = layer.mlp.activation_fn(h)
    return layer.mlp.fc2.base_forward(h)


@dataclass
class Cluster:
    cluster_id: int
    shared_layer_idx: int
    members: List[int]
    forbidden_pairs: List[int] = field(default_factory=list)


class ClusterRegistry:
    def __init__(self, num_layers: int):
        self.clusters: Dict[int, Cluster] = {}
        self._next_id = 0
        for i in range(num_layers):
            cid = self._next_id
            self.clusters[cid] = Cluster(cid, i, [i])
            self._next_id += 1

    def num_clusters(self) -> int:
        return len(self.clusters)

    def merge(self, cid_a: int, cid_b: int, shared_layer_idx: int) -> int:
        ca, cb = self.clusters[cid_a], self.clusters[cid_b]
        nid = self._next_id
        self.clusters[nid] = Cluster(nid, shared_layer_idx, ca.members + cb.members)
        del self.clusters[cid_a]
        del self.clusters[cid_b]
        self._next_id += 1
        return nid

    def summary(self) -> str:
        lines = []
        for cid, c in sorted(self.clusters.items()):
            lines.append(
                f"  cluster {cid}: members={c.members}, shared_base=layer{c.shared_layer_idx}"
            )
        return "\n".join(lines)


def assert_shared_ffn_ties(model, registry: ClusterRegistry) -> None:
    layers = get_layers(model)
    for cid, cluster in registry.clusters.items():
        rep = layers[cluster.shared_layer_idx]
        ref_fc1, ref_fc2 = rep.mlp.fc1.linear, rep.mlp.fc2.linear
        for m in cluster.members:
            blk = layers[m]
            if blk.mlp.fc1.linear is not ref_fc1 or blk.mlp.fc2.linear is not ref_fc2:
                raise RuntimeError(f"Broken FFN tie in cluster {cid} at layer {m}")


WIKITEXT_URL = "https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-raw-v1.zip"
WIKITEXT_CACHE = Path.home() / ".cache" / "wikitext-2-raw-v1"


@dataclass
class TextCorpus:
    texts: List[str]


def _ensure_wikitext() -> Path:
    WIKITEXT_CACHE.mkdir(parents=True, exist_ok=True)
    train = WIKITEXT_CACHE / "wiki.train.raw"
    if train.exists():
        return WIKITEXT_CACHE
    import urllib.request
    import zipfile
    zip_path = WIKITEXT_CACHE / "wikitext-2-raw-v1.zip"
    print("Downloading WikiText-2 raw (~4 MB)...")
    urllib.request.urlretrieve(WIKITEXT_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WIKITEXT_CACHE)
    return WIKITEXT_CACHE


def _read_wikitext_lines(path: Path) -> List[str]:
    lines = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if len(line) > 10 and not line.startswith("="):
                lines.append(line)
    return lines


def build_dataset(split: str = "train") -> TextCorpus:
    root = _ensure_wikitext()
    fname = {"train": "wiki.train.raw", "validation": "wiki.valid.raw", "test": "wiki.test.raw"}[split]
    return TextCorpus(_read_wikitext_lines(root / fname))


def get_batch(dataset: TextCorpus, tokenizer):
    texts = []
    while len(texts) < BATCH_SIZE:
        t = random.choice(dataset.texts)
        if t and len(t.strip()) > 10:
            texts.append(t.strip())
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=SEQ_LEN,
    )
    return enc.input_ids.to(DEVICE), enc.attention_mask.to(DEVICE)


@torch.no_grad()
def evaluate(model, dataset, tokenizer, num_batches: int = EVAL_BATCHES) -> float:
    was_training = model.training
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, m = get_batch(dataset, tokenizer)
        losses.append(model(x, attention_mask=m, labels=x).loss.item())
    if was_training:
        model.train()
    return sum(losses) / len(losses)


def trim_model_to_layers(model, keep_indices: List[int]) -> None:
    """Keep only selected decoder layers to reduce RAM."""
    all_layers = get_layers(model)
    kept = nn.ModuleList([all_layers[i] for i in keep_indices])
    model.model.layers = kept
    model.config.num_hidden_layers = len(keep_indices)
    gc.collect()


def count_ffn_unique_params(model) -> int:
    """Unique params in FFN fc1/fc2 only (shared pointers counted once)."""
    seen, total = set(), 0
    for layer in get_layers(model):
        for mod in (layer.mlp.fc1, layer.mlp.fc2):
            lin = mod.linear if isinstance(mod, LoRALinear) else mod
            for p in lin.parameters():
                if id(p) not in seen:
                    seen.add(id(p))
                    total += p.numel()
    return total

## Load Phi-2 and trim to 2 layers

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading full Phi-2 (this is the peak RAM moment)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Original layers: {len(model.model.layers)}")
trim_model_to_layers(model, KEEP_LAYER_INDICES)
model = model.to(DEVICE)
model.train()

num_layers = len(get_layers(model))
print(f"Kept {num_layers} layers on {DEVICE}")
print(f"Total params: {count_params_total(model):,}")
print(f"Unique params: {count_params_unique(model):,}")

Loading full Phi-2 (this is the peak RAM moment)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Original layers: 32
Kept 2 layers on cpu
Total params: 419,543,040
Unique params: 419,543,040


## Phase 0 — LoRA wrap, frozen originals, baseline loss

In [7]:
import requests
import zipfile
from pathlib import Path

WIKITEXT_URL = "https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-raw-v1.zip"
WIKITEXT_CACHE = Path("./wikitext_cache")

def _ensure_wikitext():
    WIKITEXT_CACHE.mkdir(exist_ok=True)

    extracted = WIKITEXT_CACHE / "wikitext-2-raw-v1"
    if extracted.exists():
        return extracted

    zip_path = WIKITEXT_CACHE / "wikitext-2-raw-v1.zip"

    print("Downloading WikiText-2 raw...")

    response = requests.get(
        WIKITEXT_URL,
        allow_redirects=True,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))

    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)

    # verify zip
    if not zipfile.is_zipfile(zip_path):
        with open(zip_path, "rb") as f:
            print(f.read(300))
        raise RuntimeError("Downloaded file is not a valid ZIP")

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WIKITEXT_CACHE)

    return extracted

In [8]:
rm -rf wikitext_cache

In [14]:
!wget https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-2-raw-v1.zip?download=true -O wikitext-2-raw-v1.zip
!mkdir -p wikitext_cache
!unzip wikitext-2-raw-v1.zip -d wikitext_cache

--2026-05-17 12:32:07--  https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-2-raw-v1.zip?download=true
Resolving huggingface.co (huggingface.co)... 99.86.30.85, 99.86.30.112, 99.86.30.69, ...
Connecting to huggingface.co (huggingface.co)|99.86.30.85|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-17 12:32:07 ERROR 404: Not Found.

Archive:  wikitext-2-raw-v1.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of wikitext-2-raw-v1.zip or
        wikitext-2-raw-v1.zip.zip, and cannot find wikitext-2-raw-v1.zip.ZIP, period.


In [17]:
!pip install datasets

  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 558.3 kB/s  0:00:00m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.9 MB/s  0:00:00 eta 0:00:01
Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 1.2 MB/s  0:00:37m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 920.0 kB/s  0:00:13 eta 0:00:01
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/17 [pyarrow]
    Found existing installation: fsspec 2026.4.0━━━━━━━━━━━━━━  3/17 [pyarrow]
    Uninstalling fsspec-2026.4.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/17 [pyarrow]
      Successfully uninstalled

In [18]:
from datasets import load_dataset

class TextCorpus:
    def __init__(self, texts):
        self.texts = texts

def build_dataset(split="train"):
    ds = load_dataset(
        "wikitext",
        "wikitext-2-raw-v1",
        split=split
    )

    texts = [x["text"] for x in ds if x["text"].strip()]
    return TextCorpus(texts)

In [19]:
dataset = build_dataset("train")
eval_dataset = build_dataset("validation")

for layer in get_layers(model):
    wrap_mlp_with_lora(layer, rank=LORA_RANK)

frozen_originals: Dict[int, nn.Module] = {}
for i, layer in enumerate(get_layers(model)):
    snap = copy.deepcopy(layer).to(DEVICE).eval()
    for p in snap.parameters():
        p.requires_grad = False
    frozen_originals[i] = snap

registry = ClusterRegistry(num_layers)
L_orig = evaluate(model, dataset, tokenizer)
ffn_unique_before = count_ffn_unique_params(model)

print(f"Baseline LM loss: {L_orig:.4f}")
print(f"FFN unique base params (before merge): {ffn_unique_before:,}")
print("Initial clusters:")
print(registry.summary())

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Baseline LM loss: 9.8476
FFN unique base params (before merge): 104,883,200
Initial clusters:
  cluster 0: members=[0], shared_base=layer0
  cluster 1: members=[1], shared_base=layer1


## Phase 1 — Distance between the two FFN bases

In [20]:
def cache_activations(model, dataset, tokenizer, num_batches=CALIB_BATCHES):
    model.eval()
    layers = get_layers(model)
    cache = {i: [] for i in range(len(layers))}
    hooks = []

    def make_hook(idx):
        def hook(module, inp, out):
            cache[idx].append(inp[0].detach().cpu())
        return hook

    for i, layer in enumerate(layers):
        hooks.append(layer.mlp.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        for _ in tqdm(range(num_batches), desc="cache activations"):
            x, m = get_batch(dataset, tokenizer)
            model(x, attention_mask=m)

    for h in hooks:
        h.remove()
    model.train()
    return cache


def cluster_distance(block_a, block_b, h_a, h_b) -> float:
    total, count = 0.0, 0
    with torch.no_grad():
        for h_cpu in h_a + h_b:
            h = h_cpu.to(DEVICE)
            out_a = base_mlp_forward(block_a, h)
            out_b = base_mlp_forward(block_b, h)
            total += ((out_a - out_b) ** 2).mean().item()
            count += 1
    return total / max(count, 1)


act_cache = cache_activations(model, dataset, tokenizer)
layers = get_layers(model)
d01 = cluster_distance(layers[0], layers[1], act_cache[0], act_cache[1])
print(f"Distance D(layer0, layer1) = {d01:.6f}")

cache activations:   0%|          | 0/3 [00:00<?, ?it/s]

Distance D(layer0, layer1) = 1.776759


## Phase 2 — Soft alignment (train both bases + member LoRAs)

In [21]:
def alignment_training(
    model,
    dataset,
    tokenizer,
    frozen_originals,
    rep_a: int,
    rep_b: int,
    member_layers: List[int],
    steps: int = ALIGN_STEPS,
):
    layers = get_layers(model)
    rep_a_block, rep_b_block = layers[rep_a], layers[rep_b]

    params, seen = [], set()

    def add(ps):
        for p in ps:
            if id(p) not in seen:
                seen.add(id(p))
                params.append(p)

    add(rep_a_block.mlp.fc1.linear.parameters())
    add(rep_a_block.mlp.fc2.linear.parameters())
    add(rep_b_block.mlp.fc1.linear.parameters())
    add(rep_b_block.mlp.fc2.linear.parameters())

    for m in member_layers:
        blk = layers[m]
        add([blk.mlp.fc1.A, blk.mlp.fc1.B, blk.mlp.fc2.A, blk.mlp.fc2.B])

    opt = torch.optim.AdamW(params, lr=LR_ALIGN)
    warmup = max(int(WARMUP_FRAC * steps), 1)

    for step in tqdm(range(steps), desc="alignment"):
        x, mask = get_batch(dataset, tokenizer)
        opt.zero_grad()

        out = model(x, attention_mask=mask, labels=x, output_hidden_states=True)
        loss_lm = out.loss

        loss_align = torch.tensor(0.0, device=DEVICE)
        loss_anchor = torch.tensor(0.0, device=DEVICE)

        for m in member_layers:
            h_raw = out.hidden_states[m]
            h_in = mlp_input(layers[m], h_raw)

            a_out = base_mlp_forward(rep_a_block, h_in)
            b_out = base_mlp_forward(rep_b_block, h_in)
            loss_align = loss_align + ((a_out - b_out) ** 2).mean()

            with torch.no_grad():
                target = full_mlp_forward(frozen_originals[m], h_in.detach())
            pred = full_mlp_forward(layers[m], h_in)
            loss_anchor = loss_anchor + ((pred - target) ** 2).mean()

        n = len(member_layers)
        loss_align /= n
        loss_anchor /= n
        lam = LAMBDA_MAX * min(1.0, step / warmup)
        loss = loss_lm + lam * loss_align + MU * loss_anchor
        loss.backward()
        opt.step()

        if step % max(steps // 5, 1) == 0 or step == steps - 1:
            print(
                f"  step {step:4d} | LM={loss_lm.item():.4f} "
                f"align={loss_align.item():.4f} anchor={loss_anchor.item():.4f} lam={lam:.2f}"
            )


cid_a, cid_b = 0, 1
members = registry.clusters[cid_a].members + registry.clusters[cid_b].members
rep_a, rep_b = 0, 1

pre_state = copy.deepcopy(model.state_dict())
alignment_training(model, dataset, tokenizer, frozen_originals, rep_a, rep_b, members)

L_after_align = evaluate(model, dataset, tokenizer)
delta_align = L_after_align - L_orig
print(f"After alignment: L={L_after_align:.4f}, delta vs baseline={delta_align:+.4f}")

if delta_align > THRESH_BAD:
    print(f"delta_L > {THRESH_BAD}: rejecting merge and restoring checkpoint")
    model.load_state_dict(pre_state)
    raise SystemExit("Merge rejected by loss threshold")

alignment:   0%|          | 0/150 [00:00<?, ?it/s]

  step    0 | LM=10.0604 align=2.7818 anchor=0.0000 lam=0.00
  step   30 | LM=6.8342 align=0.4481 anchor=0.0038 lam=3.00
  step   60 | LM=9.0763 align=0.6957 anchor=0.0094 lam=3.00
  step   90 | LM=5.6349 align=0.3055 anchor=0.0180 lam=3.00
  step  120 | LM=7.2340 align=0.3594 anchor=0.0145 lam=3.00
  step  149 | LM=7.0749 align=0.3073 anchor=0.0292 lam=3.00
After alignment: L=6.8808, delta vs baseline=-2.9669


## Phase 3 — Collapse to one shared FFN base

In [22]:
def collapse_to_shared_base(model, registry, cid_a: int, cid_b: int, rank: int):
    layers = get_layers(model)
    ca, cb = registry.clusters[cid_a], registry.clusters[cid_b]

    if len(ca.members) >= len(cb.members):
        rep_layer = ca.shared_layer_idx
    else:
        rep_layer = cb.shared_layer_idx

    rep = layers[rep_layer]
    ref_fc1, ref_fc2 = rep.mlp.fc1.linear, rep.mlp.fc2.linear

    for m in ca.members + cb.members:
        wrap_mlp_with_lora(layers[m], rank=rank)
        layers[m].mlp.fc1.linear = ref_fc1
        layers[m].mlp.fc2.linear = ref_fc2

    return rep_layer


rep_layer = collapse_to_shared_base(model, registry, cid_a, cid_b, LORA_RANK)
new_cid = registry.merge(cid_a, cid_b, rep_layer)
assert_shared_ffn_ties(model, registry)

ffn_unique_after = count_ffn_unique_params(model)
print("Shared FFN tie check: OK")
print(registry.summary())
print(f"FFN unique base params after merge: {ffn_unique_after:,}")
print(f"FFN base compression: {compression_ratio(ffn_unique_before, ffn_unique_after):.1f}%")

Shared FFN tie check: OK
  cluster 2: members=[0, 1], shared_base=layer0
FFN unique base params after merge: 52,441,600
FFN base compression: 50.0%


## Phase 4 — Short recovery fine-tune

In [23]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR_RECOVERY)

for step in tqdm(range(RECOVERY_STEPS), desc="recovery"):
    x, mask = get_batch(dataset, tokenizer)
    opt.zero_grad()
    loss = model(x, attention_mask=mask, labels=x).loss
    loss.backward()
    opt.step()

assert_shared_ffn_ties(model, registry)

L_final = evaluate(model, dataset, tokenizer)
print(f"Final LM loss: {L_final:.4f} (baseline {L_orig:.4f}, delta {L_final - L_orig:+.4f})")
print(f"Model unique params: {count_params_unique(model):,}")
print(f"FFN unique base params: {count_ffn_unique_params(model):,}")

recovery:   0%|          | 0/40 [00:00<?, ?it/s]

Final LM loss: 5.3865 (baseline 9.8476, delta -4.4611)
Model unique params: 367,306,240
FFN unique base params: 52,441,600


## Inspect shared weights (same `nn.Parameter` objects)

In [24]:
layers = get_layers(model)
for i in range(len(layers)):
    fc1_ptr = id(layers[i].mlp.fc1.linear.weight)
    fc2_ptr = id(layers[i].mlp.fc2.linear.weight)
    print(f"layer {i}: fc1.weight id={fc1_ptr}, fc2.weight id={fc2_ptr}")

same_fc1 = id(layers[0].mlp.fc1.linear.weight) == id(layers[1].mlp.fc1.linear.weight)
same_fc2 = id(layers[0].mlp.fc2.linear.weight) == id(layers[1].mlp.fc2.linear.weight)
print(f"\nLayers 0 and 1 share fc1 base: {same_fc1}")
print(f"Layers 0 and 1 share fc2 base: {same_fc2}")

lora0 = layers[0].mlp.fc1.A.norm().item() + layers[1].mlp.fc1.A.norm().item()
print(f"Per-layer LoRA still distinct (fc1.A norm sum): {lora0:.4f}")

layer 0: fc1.weight id=124916841841584, fc2.weight id=124916841841664
layer 1: fc1.weight id=124916841841584, fc2.weight id=124916841841664

Layers 0 and 1 share fc1 base: True
Layers 0 and 1 share fc2 base: True
Per-layer LoRA still distinct (fc1.A norm sum): 2.0095


## Optional — save trimmed merged model

In [25]:
SAVE = False  # set True to write artifacts

if SAVE:
    out_dir = "outputs/phi2_2layer_merged"
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f"Saved to {out_dir}")